# Direct Preference Optimization (DPO) using Unsloth

This notebook aligns the fine-tuned UrduGPT-v2 model using Direct Preference Optimization (DPO) on a preference dataset.

In [1]:
!pip install -q unsloth
!pip install -q trl datasets accelerate bitsandbytes peft

##Imports

In [2]:
import torch
from datasets import load_dataset

from unsloth import FastLanguageModel

from trl import (
    DPOTrainer,
    DPOConfig,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


##Load Base Model

In [3]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


##Attach LoRA

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
       "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


##Load DPO Dataset

In [5]:
dataset = load_dataset(
    'json',
    data_files='/content/dpo_dataset.jsonl',
    split='train'
)

print(dataset)
print(dataset[0])

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 55
})
{'prompt': 'پاکستان کا دارالحکومت کیا ہے؟', 'chosen': 'پاکستان کا دارالحکومت اسلام آباد ہے۔ یہ شہر ملک کے شمالی حصے میں واقع ہے اور وفاقی دارالحکومت کے طور پر کام کرتا ہے۔ اسلام آباد کی منصوبہ بندی 1960 کی دہائی میں کی گئی تھی اور اسے 1966 میں نئے دارالحکومت کے طور پر منتخب کیا گیا تھا۔ یہ شہر اپنی خوبصورت عمارتوں، پارکوں اور باغوں کے لیے مشہور ہے۔', 'rejected': 'پاکستان کا دارالحکومت کیا ہے؟پاکستان کا دارالحکومت اسلام آباد ہے۔'}


##DPOConfig

In [6]:
training_args = DPOConfig(
    output_dir="urdugpt-v2-dpo",

    num_train_epochs=3,
    learning_rate=5e-6,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    max_length=1024,
    max_prompt_length=512,

    logging_steps=5,
    save_strategy="epoch",

    fp16=True,
    bf16=False,

    beta=0.1,

    report_to="none",
)

In [7]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [8]:
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

Applying chat template to train dataset (num_proc=6):   0%|          | 0/55 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/55 [00:00<?, ? examples/s]

In [9]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 55 | Num Epochs = 3 | Total steps = 42
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
5,0.693242,-0.000511,-0.000337,0.250000,-0.000175,-467.909729,-123.959473,0.016858,-0.109275
10,0.690930,0.005132,0.000672,0.700000,0.004460,-475.454742,-101.279160,0.245456,-0.087337
15,0.675627,0.032350,-0.004082,0.947368,0.036432,-512.378113,-104.763908,0.234806,-0.070136
20,0.652946,0.075899,-0.006573,0.950000,0.082472,-479.498962,-121.118088,0.285515,-0.060826
25,0.630311,0.117247,-0.015273,1.000000,0.132520,-436.104309,-111.456619,0.087315,-0.165882
30,0.602749,0.167497,-0.021089,1.000000,0.188586,-498.975128,-112.827408,0.046589,-0.148655
35,0.582255,0.203319,-0.035967,1.000000,0.239286,-418.293884,-103.916893,0.176126,-0.160560
40,0.568183,0.248466,-0.025679,1.000000,0.274144,-478.279114,-111.867943,0.171869,-0.060191


Unsloth: Restored added_tokens_decoder metadata in urdugpt-v2-dpo/checkpoint-14/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in urdugpt-v2-dpo/checkpoint-28/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in urdugpt-v2-dpo/checkpoint-42/tokenizer_config.json.


TrainOutput(global_step=42, training_loss=0.6344730626969111, metrics={'train_runtime': 442.9677, 'train_samples_per_second': 0.372, 'train_steps_per_second': 0.095, 'total_flos': 0.0, 'train_loss': 0.6344730626969111, 'epoch': 3.0})

In [10]:
model.save_pretrained("UrduGPT-v2-DPO-LoRA")
tokenizer.save_pretrained("UrduGPT-v2-DPO-LoRA")

Unsloth: Restored added_tokens_decoder metadata in UrduGPT-v2-DPO-LoRA/tokenizer_config.json.


('UrduGPT-v2-DPO-LoRA/tokenizer_config.json',
 'UrduGPT-v2-DPO-LoRA/chat_template.jinja',
 'UrduGPT-v2-DPO-LoRA/tokenizer.json')

In [11]:
model.save_pretrained_merged(
    "UrduGPT-v2-DPO",
    tokenizer,
    save_method="merged_16bit",
)

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in UrduGPT-v2-DPO/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [02:26<00:00, 73.34s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:56<00:00, 58.28s/it]


Unsloth: Merge process complete. Saved to `/content/UrduGPT-v2-DPO`


In [13]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN_WRITE'))

In [14]:
model.push_to_hub_merged(
    "mfayazkhan/UrduGPT-v2-DPO",
    tokenizer,
    save_method="merged_16bit",
)

Unsloth: Restored added_tokens_decoder metadata in mfayazkhan/UrduGPT-v2-DPO/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...GPT-v2-DPO/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          |  608kB / 1.46GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [04:11<00:00, 125.56s/it]


Unsloth: Merge process complete. Saved to `/content/mfayazkhan/UrduGPT-v2-DPO`


In [16]:
from huggingface_hub import whoami
whoami()['name']

'mfayazkhan'